# Appendix: ResNet from Scratch

Deeper convolutional networks should be more expressive than shallower ones. After all, a deeper model includes all functions representable by the shallower model plus additional ones enabled by the extra layers. Yet in practice, simply stacking more layers onto a plain convolutional network leads to *higher* training error. This is the **degradation problem** identified by He et al. [@resnet]: the issue is not overfitting, but an optimization difficulty where deeper plain networks are harder to train.

In this appendix, we build **ResNet-18** from scratch for CIFAR-10 and demonstrate the degradation problem with a VGG-style [@vgg] baseline. The theoretical foundations for residual connections, including gradient flow through the identity path and the unraveled view of exponentially many gradient paths, were developed in [NB05](./05-activations-and-gradients.html). CNN primitives (convolution, padding, stride, pooling) were covered in [NB06](./06-cnn.html). Here we focus on the architecture and its empirical effects.

<br>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib_inline import backend_inline

import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import OneCycleLR
import torchinfo

DATASET_DIR = Path("./data").resolve()
DATASET_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 42
DEBUG = False
MATPLOTLIB_FORMAT = "png" if DEBUG else "svg"

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
warnings.simplefilter(action="ignore")
backend_inline.set_matplotlib_formats(MATPLOTLIB_FORMAT)

DEVICE = (
    torch.device("cuda:0") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print(f"Device: {DEVICE}")

## CIFAR-10

**Data.** CIFAR-10 consists of 60,000 color images of size $32 \times 32$ in 10 classes, split into 50,000 training and 10,000 test images. We apply standard data augmentation: random cropping with padding and horizontal flips for training, and per-channel normalization for both splits.

In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)
BATCH_SIZE   = 128

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_set = torchvision.datasets.CIFAR10(
    root=DATASET_DIR, train=True, download=True, transform=transform_train
)
test_set = torchvision.datasets.CIFAR10(
    root=DATASET_DIR, train=False, download=True, transform=transform_test
)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

CLASSES = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
print(f"Train: {len(train_set)}, Test: {len(test_set)}")

Visualizing a batch of training samples (without augmentation):

In [ ]:
#| code-fold: true
#| label: fig-cifar-samples
#| fig-cap: "Sample images from CIFAR-10."

viz_set = torchvision.datasets.CIFAR10(
    root=DATASET_DIR, train=True, download=False, transform=transforms.ToTensor()
)

fig, axes = plt.subplots(3, 10, figsize=(10, 3.5))
for i, ax in enumerate(axes.flat):
    img, label = viz_set[i]
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(CLASSES[label], fontsize=7)
    ax.axis("off")
fig.tight_layout()
plt.show();

## VGG-style plain network

As a baseline, we build a plain convolutional network inspired by VGG [@vgg]: stacking $3 \times 3$ convolution blocks without skip connections. Each block consists of two convolutions with batch normalization and ReLU. The network has 4 stages with progressively increasing channel counts $\{64, 128, 256, 512\}$. Spatial downsampling is performed by the first convolution of each stage (stride 2) when the resolution must decrease.

We define two variants: a **shallow** network with 1 block per stage (10 conv layers total) and a **deep** network with 3 blocks per stage (26 conv layers). If depth were purely beneficial, the deeper variant should achieve at least as good training error as the shallow one.

Defining the plain block and the plain network:

In [ ]:
class PlainBlock(nn.Module):
    """Two conv3x3-BN-ReLU layers without skip connection."""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        return out


class PlainNet(nn.Module):
    """VGG-style plain convolutional network for CIFAR-10."""
    def __init__(self, num_blocks, num_classes=10):
        super().__init__()
        self.in_channels = 64

        self.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)  # <1>
        self.bn1   = nn.BatchNorm2d(64)

        self.stage1 = self._make_stage(64,  num_blocks[0], stride=1)
        self.stage2 = self._make_stage(128, num_blocks[1], stride=2)
        self.stage3 = self._make_stage(256, num_blocks[2], stride=2)
        self.stage4 = self._make_stage(512, num_blocks[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_stage(self, out_channels, num_blocks, stride):
        layers = [PlainBlock(self.in_channels, out_channels, stride)]  # <2>
        self.in_channels = out_channels
        for _ in range(1, num_blocks):
            layers.append(PlainBlock(self.in_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

1. Initial $3 \times 3$ convolution for CIFAR-10's $32 \times 32$ inputs (rather than the $7 \times 7$ used for ImageNet's $224 \times 224$).
2. Only the first block of each stage uses `stride > 1` to halve the spatial dimensions.

Instantiating the shallow and deep variants:

In [ ]:
plain_shallow = PlainNet(num_blocks=[1, 1, 1, 1])  # 10 conv layers
plain_deep    = PlainNet(num_blocks=[3, 3, 3, 3])  # 26 conv layers

print("=== PlainNet-10 (shallow) ===")
print(f"Parameters: {sum(p.numel() for p in plain_shallow.parameters()):,}")

print("\n=== PlainNet-26 (deep) ===")
print(f"Parameters: {sum(p.numel() for p in plain_deep.parameters()):,}")

## Training

We define a self-contained training loop using AdamW with one-cycle LR scheduling, adapting the training pattern from [NB06](./06-cnn.html).

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scheduler, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)

    return total_loss / total, correct / total


@torch.inference_mode()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)

    return total_loss / total, correct / total

Full training loop with logging:

In [ ]:
def train_model(model, train_loader, test_loader, epochs, lr, device, verbose=True):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-4)
    scheduler = OneCycleLR(
        optimizer, max_lr=lr,
        steps_per_epoch=len(train_loader), epochs=epochs
    )

    history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, scheduler, device
        )
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)

        if verbose and (epoch % 10 == 0 or epoch == 1):
            print(
                f"[Epoch {epoch:>3d}/{epochs}]  "
                f"train_loss: {train_loss:.4f}  train_acc: {train_acc:.4f}  "
                f"test_loss: {test_loss:.4f}  test_acc: {test_acc:.4f}"
            )

    return history

## The degradation problem

**Q.** If a deeper network includes the shallower one as a submodel (the extra layers could simply learn the identity), why does the deeper plain network perform *worse*?

The answer is that SGD-based optimizers cannot easily drive layers toward the identity mapping. In practice, deeper plain networks converge to worse solutions, exhibiting higher training error than their shallower counterparts. This is not overfitting: the training loss itself is higher. The degradation problem was a key motivation for residual learning [@resnet].

We demonstrate this by training PlainNet-10 (shallow) and PlainNet-26 (deep) on CIFAR-10 for 100 epochs.

Training both plain networks:

In [ ]:
#| output: false
EPOCHS = 100
LR = 0.001

torch.manual_seed(RANDOM_SEED)
plain_shallow = PlainNet(num_blocks=[1, 1, 1, 1]).to(DEVICE)
hist_shallow = train_model(plain_shallow, train_loader, test_loader, EPOCHS, LR, DEVICE, verbose=False)

torch.manual_seed(RANDOM_SEED)
plain_deep = PlainNet(num_blocks=[3, 3, 3, 3]).to(DEVICE)
hist_deep = train_model(plain_deep, train_loader, test_loader, EPOCHS, LR, DEVICE, verbose=False)

**Figure.** Comparing training and test accuracy of the shallow and deep plain networks:

In [ ]:
#| code-fold: true
#| label: fig-degradation
#| fig-cap: "The degradation problem. The deeper PlainNet-26 achieves worse training and test accuracy than the shallower PlainNet-10."

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

epochs_range = range(1, EPOCHS + 1)

ax1.plot(epochs_range, hist_shallow["train_acc"], label="PlainNet-10 (shallow)", color="C0")
ax1.plot(epochs_range, hist_deep["train_acc"], label="PlainNet-26 (deep)", color="C1")
ax1.set_xlabel("epoch")
ax1.set_ylabel("train accuracy")
ax1.legend()
ax1.grid(linestyle="dotted", alpha=0.6)

ax2.plot(epochs_range, hist_shallow["test_acc"], label="PlainNet-10 (shallow)", color="C0")
ax2.plot(epochs_range, hist_deep["test_acc"], label="PlainNet-26 (deep)", color="C1")
ax2.set_xlabel("epoch")
ax2.set_ylabel("test accuracy")
ax2.legend()
ax2.grid(linestyle="dotted", alpha=0.6)

fig.tight_layout()
plt.show();

The deeper network has strictly more representational capacity, yet it achieves *lower* accuracy on both training and test sets. This confirms the degradation problem: the difficulty is in optimization, not in model capacity.

## Residual blocks

The solution from [@resnet] is to let each block learn a **residual function** $\mathcal{F}(\mathbf{x}) = \mathcal{H}(\mathbf{x}) - \mathbf{x}$ rather than the desired mapping $\mathcal{H}(\mathbf{x})$ directly. The block output becomes:

$$
\boxed{\mathbf{y} = \mathcal{F}(\mathbf{x}) + \mathbf{x}}
$$ {#eq-residual}

where $\mathcal{F}$ is the residual branch (e.g. conv $\to$ BN $\to$ ReLU $\to$ conv $\to$ BN) and the addition is the **skip connection**. If the optimal mapping is close to the identity, it is easier for the optimizer to push $\mathcal{F}$ toward zero than to learn the full identity through a stack of nonlinear layers. See [NB05](./05-activations-and-gradients.html) for the full analysis of gradient flow through the identity path and the unraveled view.

<br>

**Architecture.** The **BasicBlock** used in ResNet-18 and ResNet-34 has the following structure:

$$
\mathbf{x} \;\xrightarrow{\;\text{conv}_{3 \times 3}\;} \text{BN} \to \text{ReLU} \;\xrightarrow{\;\text{conv}_{3 \times 3}\;} \text{BN} \;\xrightarrow{\;(+\,\text{shortcut})\;} \text{ReLU}
$$

The shortcut is the identity when input and output dimensions match. When they differ (spatial downsampling or channel expansion), we apply a **projection shortcut**: a $1 \times 1$ convolution with appropriate stride followed by batch normalization. This ensures the shapes are compatible for the elementwise addition in @eq-residual.

Implementing the `BasicBlock`:

In [ ]:
class BasicBlock(nn.Module):
    """Residual block with two 3x3 convolutions and a skip connection."""
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels, out_channels, 3,
            stride=stride, padding=1, bias=False   # <1>
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(
            out_channels, out_channels, 3,
            stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample               # <2>

    def forward(self, x):
        identity = x

        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        if self.downsample is not None:
            identity = self.downsample(x)           # <3>

        out += identity                             # <4>
        return F.relu(out)

1. Bias is omitted since the subsequent `BatchNorm2d` absorbs any bias term.
2. The `downsample` module (a $1 \times 1$ conv + BN) is applied to the shortcut path when input/output shapes differ.
3. Project the identity to match the residual branch output shape.
4. The skip connection: elementwise addition before the final ReLU.

## Building ResNet-18

ResNet-18 consists of 4 stages with channel counts $\{64, 128, 256, 512\}$ and 2 `BasicBlock`s per stage. For ImageNet ($224 \times 224$ inputs), the original architecture uses a $7 \times 7$ convolution with stride 2 followed by a $3 \times 3$ max pool as the stem. For CIFAR-10 ($32 \times 32$ inputs), we follow the common practice of replacing this with a single $3 \times 3$ convolution with stride 1 and no pooling, so that the spatial resolution is preserved in the first stage.

The `_make_layer` method handles the construction of each stage. The first block of a stage may downsample (stride 2) and change channels, while subsequent blocks maintain the same dimensions.

Defining the ResNet architecture:

In [ ]:
class ResNet(nn.Module):
    """ResNet for CIFAR-10 (3x3 stem, no initial max pool)."""
    def __init__(self, block, num_blocks, num_classes=10):
        super().__init__()
        self.in_channels = 64

        # CIFAR-10 stem: 3x3 conv instead of 7x7 + maxpool
        self.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(block, 64,  num_blocks[0], stride=1)  # <1>
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)  # <2>
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        downsample = None
        if stride != 1 or self.in_channels != out_channels * block.expansion:  # <3>
            downsample = nn.Sequential(
                nn.Conv2d(
                    self.in_channels, out_channels * block.expansion,
                    kernel_size=1, stride=stride, bias=False
                ),
                nn.BatchNorm2d(out_channels * block.expansion),
            )

        layers = [block(self.in_channels, out_channels, stride, downsample)]
        self.in_channels = out_channels * block.expansion
        for _ in range(1, num_blocks):
            layers.append(block(self.in_channels, out_channels))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))  # (B, 64, 32, 32)
        x = self.layer1(x)                    # (B, 64, 32, 32)
        x = self.layer2(x)                    # (B, 128, 16, 16)
        x = self.layer3(x)                    # (B, 256, 8, 8)
        x = self.layer4(x)                    # (B, 512, 4, 4)
        x = self.avgpool(x)                   # (B, 512, 1, 1)
        x = torch.flatten(x, 1)               # (B, 512)
        return self.fc(x)


def resnet18(num_classes=10):
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes)

1. Stage 1 uses stride 1 since the $32 \times 32$ spatial resolution is already small.
2. Stages 2--4 use stride 2 in the first block to halve spatial dimensions: $32 \to 16 \to 8 \to 4.$
3. A projection shortcut is needed whenever (a) the stride changes spatial dimensions or (b) the channel count changes.

Model summary:

In [ ]:
model = resnet18()
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training ResNet-18

**Training.** We train ResNet-18 on CIFAR-10 for 100 epochs using the same hyperparameters as the plain networks.

In [ ]:
#| output: false
torch.manual_seed(RANDOM_SEED)
resnet18_model = resnet18().to(DEVICE)
hist_resnet = train_model(resnet18_model, train_loader, test_loader, EPOCHS, LR, DEVICE, verbose=False)

**Figure.** Comparing all three models:

In [ ]:
#| code-fold: true
#| label: fig-resnet-vs-plain
#| fig-cap: "ResNet-18 achieves higher training and test accuracy than both plain network variants, despite being deeper than PlainNet-26."

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

epochs_range = range(1, EPOCHS + 1)

ax1.plot(epochs_range, hist_shallow["train_acc"], label="PlainNet-10", color="C0")
ax1.plot(epochs_range, hist_deep["train_acc"], label="PlainNet-26", color="C1")
ax1.plot(epochs_range, hist_resnet["train_acc"], label="ResNet-18", color="C2", linewidth=2)
ax1.set_xlabel("epoch")
ax1.set_ylabel("train accuracy")
ax1.legend()
ax1.grid(linestyle="dotted", alpha=0.6)

ax2.plot(epochs_range, hist_shallow["test_acc"], label="PlainNet-10", color="C0")
ax2.plot(epochs_range, hist_deep["test_acc"], label="PlainNet-26", color="C1")
ax2.plot(epochs_range, hist_resnet["test_acc"], label="ResNet-18", color="C2", linewidth=2)
ax2.set_xlabel("epoch")
ax2.set_ylabel("test accuracy")
ax2.legend()
ax2.grid(linestyle="dotted", alpha=0.6)

fig.tight_layout()
plt.show();

ResNet-18, with 18 convolutional layers, outperforms both the shallow PlainNet-10 and the deeper PlainNet-26 on training *and* test accuracy. This demonstrates that skip connections solve the degradation problem and enable effective training of deeper networks.

## Ablation: disabling skip connections

To verify that the skip connections are responsible for the improvement, we train the same ResNet-18 architecture with the identity shortcut zeroed out. This reduces the `BasicBlock` to a plain block, isolating the effect of the skip connection from other architectural choices (e.g. projection layers).

Defining a `BasicBlock` variant with no skip:

In [ ]:
class BasicBlockNoSkip(nn.Module):
    """Same as BasicBlock but with skip connection disabled."""
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels, out_channels, 3,
            stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(
            out_channels, out_channels, 3,
            stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out)  # no skip connection


def resnet18_no_skip(num_classes=10):
    return ResNet(BasicBlockNoSkip, [2, 2, 2, 2], num_classes)

Training the ablated model:

In [ ]:
#| output: false
torch.manual_seed(RANDOM_SEED)
resnet18_noskip_model = resnet18_no_skip().to(DEVICE)
hist_noskip = train_model(
    resnet18_noskip_model, train_loader, test_loader, EPOCHS, LR, DEVICE, verbose=False
)

**Figure.** Ablation comparing ResNet-18 with and without skip connections:

In [ ]:
#| code-fold: true
#| label: fig-ablation
#| fig-cap: "Removing skip connections from ResNet-18 degrades both training and test accuracy, confirming that the skip connections are the key ingredient."

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

epochs_range = range(1, EPOCHS + 1)

ax1.plot(epochs_range, hist_resnet["train_acc"], label="ResNet-18", color="C2", linewidth=2)
ax1.plot(epochs_range, hist_noskip["train_acc"], label="ResNet-18 (no skip)", color="C3", linestyle="--")
ax1.set_xlabel("epoch")
ax1.set_ylabel("train accuracy")
ax1.legend()
ax1.grid(linestyle="dotted", alpha=0.6)

ax2.plot(epochs_range, hist_resnet["test_acc"], label="ResNet-18", color="C2", linewidth=2)
ax2.plot(epochs_range, hist_noskip["test_acc"], label="ResNet-18 (no skip)", color="C3", linestyle="--")
ax2.set_xlabel("epoch")
ax2.set_ylabel("test accuracy")
ax2.legend()
ax2.grid(linestyle="dotted", alpha=0.6)

fig.tight_layout()
plt.show();

Without skip connections, the model degrades significantly. The only difference between the two models is the identity shortcut in the `BasicBlock`. This confirms that [skip connections are the key mechanism]{.mark} enabling effective training of deep convolutional networks.

## Conclusion

We verified three key observations from [@resnet]:

1. **Degradation.** Deeper plain networks achieve worse training accuracy than shallower ones, confirming that depth alone is insufficient.
2. **Residual learning.** Adding skip connections to form residual blocks solves the degradation problem and enables deeper networks to train effectively.
3. **Skip connections are essential.** Ablating the skip connection from ResNet-18 restores the degradation, confirming that the identity shortcut is the active ingredient.

**Remark.** The `BasicBlock` used here is the building block for ResNet-18 and ResNet-34. For deeper variants (ResNet-50, ResNet-101, ResNet-152), a **Bottleneck** block is used instead: $1 \times 1$ conv (reduce channels) $\to$ $3 \times 3$ conv $\to$ $1 \times 1$ conv (expand channels). This reduces computational cost while maintaining depth. These deeper variants also use `expansion = 4` in the bottleneck, so the output channels are $4\times$ the internal channels.

<br>

**Remark.** For CIFAR-10, the stem layer is a $3 \times 3$ convolution with stride 1 instead of the ImageNet stem ($7 \times 7$ conv, stride 2, followed by $3 \times 3$ max pool). This preserves the spatial resolution at $32 \times 32$ which is already much smaller than ImageNet's $224 \times 224.$ The pretrained ResNet used for transfer learning in [NB06](./06-cnn.html) uses the full ImageNet stem.

:::{.callout-note}
In [NB06](./06-cnn.html), we used a pretrained `resnet18` as a frozen feature extractor for transfer learning. The architecture built here is the same model, trained from scratch on CIFAR-10 instead of ImageNet.

:::

---

■